# Petit Transformer de texte avec Tiny Shakespeare

Notebook léger et reproductible pour Kaggle ou un GPU local.
Le fichier texte attendu est `./data/tiny_shakespeare.txt`.
On charge un tokenizer Hugging Face, on entraîne un petit GPT depuis zéro,
puis on suit la loss et la perplexité uniquement.


In [4]:
%pip install --quiet torch transformers datasets tokenizers accelerate matplotlib

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import math
import random

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, GPT2Config, GPT2LMHeadModel, Trainer, TrainingArguments

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_PATH = Path("./data/tiny_shakespeare.txt")
OUTPUT_DIR = Path("./outputs/tiny_shakespeare_gpt")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 128
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01

N_LAYER = 4
N_HEAD = 4
N_EMBD = 256

print("CUDA available:", torch.cuda.is_available())
print("Data path:", DATA_PATH)

CUDA available: False
Data path: data\tiny_shakespeare.txt


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Place the Tiny Shakespeare text file at: {DATA_PATH}")

raw_dataset = load_dataset("text", data_files=str(DATA_PATH))
print(raw_dataset)  # affiche les clés disponibles
raw_dataset = raw_dataset["train"] if "train" in raw_dataset else raw_dataset[list(raw_dataset.keys())[0]]
split_80_20 = raw_dataset.train_test_split(test_size=0.20, seed=SEED)
train_raw = split_80_20["train"]
temp_raw = split_80_20["test"]
split_10_10 = temp_raw.train_test_split(test_size=0.50, seed=SEED)
val_raw = split_10_10["train"]
test_raw = split_10_10["test"]

print("Train rows:", len(train_raw))
print("Validation rows:", len(val_raw))
print("Test rows:", len(test_raw))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

train_tokenized = train_raw.map(
    lambda batch: tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["text"],
)
val_tokenized = val_raw.map(
    lambda batch: tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["text"],
)
test_tokenized = test_raw.map(
    lambda batch: tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["text"],
)

print(train_tokenized[0])
print("Tokenizer vocab size:", tokenizer.vocab_size)

In [ ]:
config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=MAX_LENGTH,
    n_ctx=MAX_LENGTH,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    resid_pdrop=0.1,
    embd_pdrop=0.1,
    attn_pdrop=0.1,
)

model = GPT2LMHeadModel(config)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Build TrainingArguments in a way that's robust across transformers versions
training_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    logging_steps=25,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

training_args = TrainingArguments(**training_kwargs)
# Optional params: set them if the installed TrainingArguments supports them
optional_attrs = {
    "evaluation_strategy": "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "remove_unused_columns": False,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
}
for k, v in optional_attrs.items():
    try:
        setattr(training_args, k, v)
    except Exception:
        pass

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)
# attach tokenizer for compatibility with newer code paths
try:
    trainer.tokenizer = tokenizer
except Exception:
    pass

print(model)

In [ ]:
train_result = trainer.train()
val_results = trainer.evaluate()
test_results = trainer.evaluate(test_tokenized, metric_key_prefix="test")

val_loss = val_results["eval_loss"]
test_loss = test_results["test_loss"]
val_perplexity = math.exp(val_loss)
test_perplexity = math.exp(test_loss)

print(f"Validation loss: {val_loss:.4f}")
print(f"Validation perplexity: {val_perplexity:.2f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test perplexity: {test_perplexity:.2f}")

In [ ]:
model = trainer.model
model.to(training_args.device)
model.eval()

prompts = [
    "To be, or not to be",
    "The king said",
    "Tomorrow and tomorrow",
]

for prompt in prompts:
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(training_args.device)
    generated_ids = model.generate(
        input_ids,
        max_new_tokens=80,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
    )
    print("\nPROMPT:", prompt)
    print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))